# 第7章 资产定价与因子模型 —— 配套代码

对应正文 `book/part2/07-factor-models.md`。

> **运行前准备**：请先在终端执行 `uv run python scripts/make_sample_data.py` 生成内置示例数据。

## 数据说明

本 notebook 全部使用内置数据集，**完全离线可跑**：

1. **股票价格**（`load_sample_prices()`）：4 只 A 股风格资产（BANK/LIQUOR/TECH/UTILITY），约 750 个交易日。
2. **市场数据**（`load_market()`）：真实市场指数日收益与无风险利率（`rf_daily`），用于 CAPM 回归。4 只股票对该指数有真实 beta。
3. **因子数据**（`load_factors()`）：749 行日度数据，列 `MKT`（市场超额收益，真实）、`HML`（价值−成长多空构造，真实）、`SMB` 和 `MOM`（标注的合成示意因子）。

## 演示内容

1. 环境初始化与数据加载
2. 使用真实市场指数构造超额收益
3. 单因子 CAPM 回归（OLS 摘要解读）
4. 四只股票 Beta 对比汇总
5. SML 可视化 + 散点拟合图
6. Beta 条形图 + R2 方差分解
7. 加载内置因子数据集
8. FF 三因子回归（真实 MKT+HML，合成 SMB）
9. 各股票 FF3 汇总（HML 载荷价值/成长分化）
10. Carhart 四因子回归（加入合成 MOM）
11. 因子相关性与 VIF 多重共线性诊断
12. 业绩归因分解（CAPM，真实数据）
13. 因子累计收益时序图
14. 习题参考解答：习题 7.1
15. 习题参考解答：习题 7.2
16. 习题参考解答：习题 7.3
17. 习题参考解答：习题 7.4

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# Cell 1：环境初始化与数据加载
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import statsmodels.api as sm
from scipy import stats

from fds import load_sample_prices, load_market, load_factors, daily_returns, set_chinese_font

set_chinese_font()

prices  = load_sample_prices()
rets    = daily_returns(prices)   # 简单日收益率
market  = load_market()           # 真实市场指数与无风险利率
factors = load_factors()          # 内置因子数据集

TICKERS      = list(rets.columns)  # ['BANK', 'LIQUOR', 'TECH', 'UTILITY']
TRADING_DAYS = 252

print(f'价格数据维度：{prices.shape}')
print(f'收益率数据维度：{rets.shape}')
print(f'交易日范围：{rets.index[0].date()} 至 {rets.index[-1].date()}')
print()
print('=== 市场数据（load_market）样例 ===')
print(market.tail(3))
print()
print('=== 因子数据（load_factors）样例 ===')
print(factors.tail(3))

## 7.2 使用真实市场指数构造超额收益

`load_market()` 提供真实市场指数日收益率与无风险利率。
本章直接用该指数作为市场组合，**不再使用 4 只股票等权代理**。

CAPM 时序回归方程：

$$r_{i,t} - r_{f,t} = \alpha_i + \beta_i (r_{m,t} - r_{f,t}) + \varepsilon_{i,t}$$

其中 $r_{f,t}$ 取 `rf_daily`（日度无风险利率），$r_{m,t}$ 取 `index_return`（真实指数日收益率）。

In [ ]:
# Cell 2：使用真实市场指数与无风险利率构造超额收益

# 对齐日期索引（取股票收益率与市场数据的交集）
common_idx  = rets.index.intersection(market.index)
rets_c      = rets.loc[common_idx]
market_c    = market.loc[common_idx]

rf_daily    = market_c['rf_daily']       # 日度无风险利率（真实）
mkt_ret     = market_c['index_return']   # 真实指数日收益率
mkt_excess  = mkt_ret - rf_daily         # 市场超额收益（MKT 因子）
mkt_excess.name = 'MKT'

excess_rets = rets_c.sub(rf_daily, axis=0)  # 各股票超额收益

RF_ANNUAL = rf_daily.mean() * TRADING_DAYS

print('=== 市场指数统计（真实数据）===')
print(f'日均收益：{mkt_ret.mean():.4f}')
print(f'日波动率：{mkt_ret.std():.4f}')
print(f'年化收益：{mkt_ret.mean() * TRADING_DAYS:.2%}')
print(f'年化波动：{mkt_ret.std() * np.sqrt(TRADING_DAYS):.2%}')
print(f'年化无风险利率（均值）：{RF_ANNUAL:.2%}')
print(f'市场风险溢价（年化）：{mkt_excess.mean() * TRADING_DAYS:.2%}')
print(f'共用交易日数：{len(common_idx)}')
print()
print('各股票日超额收益（前 3 行）')
excess_rets.head(3)

## 7.3 单因子 CAPM 回归：以 TECH 为例

用 `statsmodels.OLS` 估计 CAPM，重点解读：
- `const`（截距）= $\alpha$，CAPM 下应为 0
- 斜率 = $\beta$，衡量市场敏感度
- $R^2$ = 系统性风险占总风险比例
- $t$-统计量和 $p$-值：检验系数是否显著异于 0

In [ ]:
# Cell 3：TECH 单因子 CAPM 回归（详细摘要）

y = excess_rets['TECH']
X = sm.add_constant(mkt_excess)
X.columns = ['alpha_const', 'MKT']

result_tech = sm.OLS(y, X).fit()
print(result_tech.summary())

In [ ]:
# Cell 4：四只股票 CAPM 回归汇总（基于真实市场指数）

capm_results = {}
X_mkt = sm.add_constant(mkt_excess)
X_mkt.columns = ['alpha_const', 'MKT']

rows = []
for ticker in TICKERS:
    res = sm.OLS(excess_rets[ticker], X_mkt).fit()
    capm_results[ticker] = res
    rows.append({
        '股票': ticker,
        'Alpha年化': round(res.params['alpha_const'] * TRADING_DAYS, 4),
        'Alpha_t': round(res.tvalues['alpha_const'], 3),
        'Alpha_p': round(res.pvalues['alpha_const'], 4),
        'Beta': round(res.params['MKT'], 4),
        'Beta_t': round(res.tvalues['MKT'], 3),
        'R2': round(res.rsquared, 4),
        '特质风险': round(1 - res.rsquared, 4),
    })

df_capm = pd.DataFrame(rows).set_index('股票')
print('=== 四只股票 CAPM 回归结果汇总（真实市场指数）===')
print(df_capm.to_string())
print()
print('预期 Beta（真实数据验证）：UTILITY≈0.24, BANK≈0.39, LIQUOR≈1.09, TECH≈1.93')
print('Beta_t 大：市场因子对该股票的解释力显著')
print('Alpha_p < 0.05：在 5% 水平下 alpha 显著异于 0（CAPM 预测为 0）')

## 7.4 CAPM 可视化：SML 与散点图

**证券市场线（SML）**：横轴为 $\beta$，纵轴为年化超额收益。
CAPM 预测所有资产都应落在 SML 上；实际位置偏离代表正/负 alpha。

使用真实市场指数，四只股票的 beta 差异显著：
UTILITY（防御性公用事业）最低，TECH（科技成长）最高，符合行业直觉。

In [ ]:
# Cell 5：SML 可视化 + TECH 散点回归图

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# -- 左图：SML --
ax1 = axes[0]
mkt_premium_ann   = mkt_excess.mean() * TRADING_DAYS
betas             = {t: capm_results[t].params['MKT'] for t in TICKERS}
actual_excess_ann = excess_rets.mean() * TRADING_DAYS

beta_range = np.linspace(0, max(betas.values()) * 1.25, 100)
ax1.plot(beta_range, beta_range * mkt_premium_ann, 'k--', lw=2,
         label=f'SML（市场溢价={mkt_premium_ann:.1%}/年）')

for i, ticker in enumerate(TICKERS):
    beta_i    = betas[ticker]
    capm_pred = beta_i * mkt_premium_ann
    actual    = actual_excess_ann[ticker]
    ax1.annotate('', xy=(beta_i, actual), xytext=(beta_i, capm_pred),
                 arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5))
    ax1.scatter(beta_i, actual, color=colors[i], s=100, zorder=5, label=ticker)
    alpha_ann = capm_results[ticker].params['alpha_const'] * TRADING_DAYS
    ax1.annotate(f'{ticker}\na={alpha_ann:.1%}',
                 xy=(beta_i, actual), xytext=(8, 5),
                 textcoords='offset points', color=colors[i], fontsize=9)

ax1.axhline(0, color='gray', lw=0.8, linestyle=':')
ax1.set_xlabel('Beta (β)')
ax1.set_ylabel('年化超额收益')
ax1.set_title('证券市场线（SML）与实际收益对比\n（真实市场指数）')
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax1.legend(fontsize=9)

# -- 右图：TECH 散点 + 回归线 --
ax2 = axes[1]
res_t  = capm_results['TECH']
x_vals = mkt_excess.values
y_vals = excess_rets['TECH'].values
ax2.scatter(x_vals, y_vals, alpha=0.3, s=15, color='#2ca02c', label='日度数据点')

x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
y_line = res_t.params['alpha_const'] + res_t.params['MKT'] * x_line
b_str  = f"{res_t.params['MKT']:.2f}"
a_str  = f"{res_t.params['alpha_const']*TRADING_DAYS:.1%}"
ax2.plot(x_line, y_line, 'r-', lw=2,
         label=f'CAPM fit: a={a_str}/年, β={b_str}')

ax2.axhline(0, color='gray', lw=0.8, linestyle=':')
ax2.axvline(0, color='gray', lw=0.8, linestyle=':')
ax2.set_xlabel('市场超额收益（日，真实指数）')
ax2.set_ylabel('TECH 超额收益（日）')
ax2.set_title(f'TECH CAPM 散点图（R2={res_t.rsquared:.3f}）')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Beta 汇总（真实市场指数）：')
for t in TICKERS:
    print(f'  {t}: beta = {betas[t]:.4f}')

In [ ]:
# Cell 6：Beta 条形图 + R2 方差分解

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# 左图：Beta 对比
ax1 = axes[0]
beta_vals = [capm_results[t].params['MKT'] for t in TICKERS]
bars = ax1.bar(TICKERS, beta_vals, color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)
ax1.axhline(1.0, color='black', lw=1.5, linestyle='--', label='beta=1（市场基准）')
ax1.set_title('各股票 Beta（对真实市场指数）')
ax1.set_ylabel('Beta')
ax1.set_ylim(0, max(beta_vals) * 1.25)
ax1.legend()
for bar, val in zip(bars, beta_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# 右图：R2 分解
ax2 = axes[1]
r2_vals   = [capm_results[t].rsquared for t in TICKERS]
idio_vals = [1 - r for r in r2_vals]
x = np.arange(len(TICKERS))
width = 0.5
bar1 = ax2.bar(x, r2_vals,   width, label='系统性风险 (R2)',  color='steelblue', alpha=0.85)
ax2.bar(x, idio_vals, width, bottom=r2_vals,
        label='特质风险 (1-R2)', color='salmon', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(TICKERS)
ax2.set_title('方差分解：系统性风险 vs 特质风险')
ax2.set_ylabel('占总方差比例')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.set_ylim(0, 1.0)
ax2.legend()
for bar, val in zip(bar1, r2_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val/2,
             f'{val:.1%}', ha='center', va='center',
             fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
plt.show()
print('R2 越高，CAPM 对该股票解释力越强；1-R2 越高，特质风险越大')

## 7.5 多因子回归：加载内置因子数据集

> **数据说明**：`load_factors()` 返回内置因子数据集（749 行日度数据）。
> - `MKT`：市场超额收益，**基于真实数据构造，结论可信**；
> - `HML`：价值−成长因子，由 `(BANK+UTILITY)/2 − (TECH+LIQUOR)/2` 多空构造，
>   与 FF 方法一致，**基于真实数据，结论可信**；
> - `SMB`、`MOM`：**合成示意因子**（本股票池仅4只、无市值/动量数据，无法真实构造），
>   仅用于演示多因子回归的操作流程，不代表真实结论。
>   如需真实因子数据，请参见附录C数据字典，或从 CSMAR、Wind 获取。

FF 三因子回归方程：

$$r_{i,t} - r_{f,t} = \alpha_i + \beta^{MKT}_i MKT_t
+ \beta^{SMB}_i SMB_t + \beta^{HML}_i HML_t + \varepsilon_{i,t}$$

In [ ]:
# Cell 7：加载并展示内置因子数据集

# 对齐因子与股票收益率的日期索引
common_f = excess_rets.index.intersection(factors.index)
factors_c = factors.loc[common_f]
excess_f  = excess_rets.loc[common_f]

print('=== 内置因子数据集（load_factors）===')
print(f'维度：{factors_c.shape}')
print(f'日期范围：{factors_c.index[0].date()} 至 {factors_c.index[-1].date()}')
print()
ann_f  = factors_c.mean() * TRADING_DAYS
ann_v  = factors_c.std()  * np.sqrt(TRADING_DAYS)
sharpe = ann_f / ann_v
summary_df = pd.DataFrame({
    '年化均值': ann_f.round(4),
    '年化波动': ann_v.round(4),
    '夏普': sharpe.round(4),
    '数据性质': ['真实', '合成示意', '真实', '合成示意'],
})
print(summary_df)
print()
print('MKT、HML 基于真实数据构造；SMB、MOM 为合成示意因子')

In [ ]:
# Cell 8：FF 三因子回归（等权组合为被解释变量）
# MKT、HML 为真实数据；SMB 为合成示意因子

y_port = excess_f.mean(axis=1)   # 等权组合超额收益

X_ff3 = sm.add_constant(factors_c[['MKT', 'SMB', 'HML']])
X_ff3.columns = ['alpha_const', 'MKT', 'SMB', 'HML']

result_ff3 = sm.OLS(y_port, X_ff3).fit()
print('=== FF 三因子回归结果（等权组合）===')
print(result_ff3.summary())
print()
print('MKT 和 HML 载荷来自真实数据，有经济学意义；SMB 为合成示意因子')

In [ ]:
# Cell 9：各股票 FF 三因子回归汇总
# 重点：HML 载荷的价值/成长分化——本章核心结论

ff3_results = {}
ff3_rows = []
for ticker in TICKERS:
    res3 = sm.OLS(excess_f[ticker], X_ff3).fit()
    ff3_results[ticker] = res3
    ff3_rows.append({
        '股票': ticker,
        'a_年化': round(res3.params['alpha_const'] * TRADING_DAYS, 4),
        'a_t': round(res3.tvalues['alpha_const'], 3),
        'b_MKT': round(res3.params['MKT'], 4),
        'b_SMB': round(res3.params['SMB'], 4),
        'b_HML': round(res3.params['HML'], 4),
        'HML_t': round(res3.tvalues['HML'], 3),
        'R2_FF3': round(res3.rsquared, 4),
        'R2_CAPM': round(capm_results[ticker].rsquared, 4),
        'R2提升': round(res3.rsquared - capm_results[ticker].rsquared, 4),
    })

df_ff3 = pd.DataFrame(ff3_rows).set_index('股票')
print('=== 四只股票 FF 三因子回归汇总 ===')
print(df_ff3.to_string())
print()
print('【核心结论】HML 载荷的价值/成长分化（真实数据）：')
print('  BANK, UTILITY：HML 载荷为正（价值股），t ≈ +10, +6，高度显著')
print('  LIQUOR, TECH：HML 载荷为负（成长股），t ≈ -17, -27，高度显著')
print('  这一正负分化符合 Fama-French 因子定义，结论可信')

## 7.6 Carhart 四因子：加入动量

> **注意**：`MOM` 为合成示意因子，仅演示四因子回归的操作流程。
> `MKT`、`HML` 为真实数据，其载荷结论有效。

$$r_{i,t} - r_{f,t} = \alpha_i + \beta^{MKT} MKT_t
+ \beta^{SMB} SMB_t + \beta^{HML} HML_t + \beta^{MOM} MOM_t + \varepsilon_{i,t}$$

In [ ]:
# Cell 10：Carhart 四因子回归（MOM 为合成示意因子）

X_c4 = sm.add_constant(factors_c[['MKT', 'SMB', 'HML', 'MOM']])
X_c4.columns = ['alpha_const', 'MKT', 'SMB', 'HML', 'MOM']

c4_rows = []
for ticker in TICKERS:
    res4 = sm.OLS(excess_f[ticker], X_c4).fit()
    c4_rows.append({
        '股票': ticker,
        'a_年化': round(res4.params['alpha_const'] * TRADING_DAYS, 4),
        'a_t': round(res4.tvalues['alpha_const'], 3),
        'a_p': round(res4.pvalues['alpha_const'], 4),
        'b_MKT': round(res4.params['MKT'], 4),
        'b_SMB': round(res4.params['SMB'], 4),
        'b_HML': round(res4.params['HML'], 4),
        'b_MOM': round(res4.params['MOM'], 4),
        'R2_C4': round(res4.rsquared, 4),
        'AIC': round(res4.aic, 2),
    })

print('=== 四只股票 Carhart 四因子回归汇总 ===')
print(pd.DataFrame(c4_rows).set_index('股票').to_string())
print()
print('alpha 检验：a_p < 0.05 说明在 5% 水平下存在显著超额 alpha')
print('MOM 因子为合成示意，其载荷仅演示操作流程，不代表真实 A 股动量结论')

## 7.7 因子相关性与多重共线性诊断

多因子回归的前提：各因子之间相关性不能过高，否则系数估计不稳定。

**方差膨胀因子 VIF**：
$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$
其中 $R_j^2$ 是用其他所有因子回归第 $j$ 个因子的 $R^2$。
VIF < 5 正常；> 10 视为严重多重共线性。

In [ ]:
# Cell 11：因子相关性 + VIF 计算（内置因子数据集）

factor_names = ['MKT', 'SMB', 'HML', 'MOM']
F = factors_c[factor_names]

corr_matrix = F.corr()
print('=== 因子相关性矩阵（内置数据集）===')
print(corr_matrix.round(4).to_string())
print()

def compute_vif(X_df):
    rows = []
    for col in X_df.columns:
        X_others = sm.add_constant(X_df.drop(columns=[col]))
        r2  = sm.OLS(X_df[col], X_others).fit().rsquared
        vif = 1 / (1 - r2) if r2 < 1 else float('inf')
        rows.append({'因子': col, 'R2': round(r2, 4), 'VIF': round(vif, 4)})
    return pd.DataFrame(rows).set_index('因子')

print('=== VIF（内置因子数据集）===')
print(compute_vif(F).to_string())
print()
print('MKT、HML 基于真实数据，若两者相关性高则需关注共线性')
print('真实 A 股中 SMB 与 HML 可能存在较高相关性，需特别检验')

# 热图
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson 相关系数')
ax.set_xticks(range(len(factor_names)))
ax.set_yticks(range(len(factor_names)))
ax.set_xticklabels(factor_names)
ax.set_yticklabels(factor_names)
ax.set_title('因子相关性热图（内置数据集）')
for i in range(len(factor_names)):
    for j in range(len(factor_names)):
        val = corr_matrix.iloc[i, j]
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=11,
                color='white' if abs(val) > 0.5 else 'black')
plt.tight_layout()
plt.show()

## 7.8 业绩归因分解（CAPM，真实数据）

$$\underbrace{r_i - r_f}_{\text{总超额收益}}
= \underbrace{\alpha_i}_{\text{真实 alpha}}
+ \underbrace{\beta_i \cdot \overline{MKT}}_{\text{市场因子贡献}}$$

多因子版本将各因子贡献逐项拆解，便于判断收益来源于选股能力还是因子暴露。

In [ ]:
# Cell 12：业绩归因（CAPM 单因子，真实市场指数）

mkt_mean_ann = mkt_excess.mean() * TRADING_DAYS

attr_rows = []
for ticker in TICKERS:
    res       = capm_results[ticker]
    alpha_ann = res.params['alpha_const'] * TRADING_DAYS
    beta_val  = res.params['MKT']
    mkt_ctr   = beta_val * mkt_mean_ann
    total     = excess_rets[ticker].mean() * TRADING_DAYS
    attr_rows.append({
        '股票': ticker,
        '总超额收益年化': round(total, 4),
        'Alpha年化': round(alpha_ann, 4),
        '市场因子贡献': round(mkt_ctr, 4),
        '误差': round(total - alpha_ann - mkt_ctr, 8),
    })

df_attr = pd.DataFrame(attr_rows).set_index('股票')
print('=== CAPM 单因子业绩归因（年化，真实市场指数）===')
print(df_attr.round(4).to_string())
print(f'\n样本内年化市场超额收益（真实指数）：{mkt_mean_ann:.2%}')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(TICKERS))
alpha_vals = [r['Alpha年化'] for r in attr_rows]
mkt_vals   = [r['市场因子贡献'] for r in attr_rows]
ax.bar(x, alpha_vals, 0.5, label='Alpha（选股贡献）',
       color='#2ca02c', alpha=0.85)
ax.bar(x, mkt_vals, 0.5, bottom=alpha_vals,
       label='市场因子贡献（Beta 暴露）', color='steelblue', alpha=0.85)
for i, total in enumerate(df_attr['总超额收益年化']):
    ax.text(i, total + 0.002, f'{total:.1%}', ha='center',
            fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(TICKERS)
ax.set_ylabel('年化超额收益')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('CAPM 业绩归因：Alpha vs 市场因子贡献（真实市场指数）')
ax.axhline(0, color='black', lw=0.8)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13：因子累计收益时序图（内置数据集）

factor_labels = [
    ('MKT', '市场超额收益（真实数据）'),
    ('SMB', 'SMB 规模因子（合成示意数据）'),
    ('HML', 'HML 价值因子（真实数据）'),
    ('MOM', 'MOM 动量因子（合成示意数据）'),
]
factor_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
for ax, (fname, flabel), color in zip(axes, factor_labels, factor_colors):
    cumret  = (1 + factors_c[fname]).cumprod() - 1
    ann_ret = factors_c[fname].mean() * TRADING_DAYS
    ax.plot(factors_c.index, cumret, color=color, lw=1.2)
    ax.fill_between(factors_c.index, cumret, 0, color=color, alpha=0.15)
    ax.set_title(f'{flabel} | 年化 = {ann_ret:.2%}', fontsize=10)
    ax.set_ylabel('累计收益')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
axes[-1].set_xlabel('日期')
fig.suptitle('因子累计收益时序图（内置数据集）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7.9 本章小结

| 模型 | 因子数 | 用途 | 关键限制 |
|---|---|---|---|
| CAPM | 1（MKT） | beta 估计、基准收益 | 低 R2，忽略规模/价值/动量 |
| FF 三因子 | 3（+SMB, HML） | 风险归因、alpha 检验 | 忽略动量 |
| Carhart 四因子 | 4（+MOM） | 基金绩效评估 | 动量在 A 股有争议 |
| FF 五因子 | 5（+RMW, CMA） | 全面风险分解 | HML 冗余，未纳入动量 |

**本章真实结论**：
- CAPM beta（真实市场指数）：UTILITY 0.24 < BANK 0.39 < LIQUOR 1.09 < TECH 1.93，符合行业直觉；
- HML 载荷（真实数据）：BANK/UTILITY 正载荷（价值股）vs LIQUOR/TECH 负载荷（成长股），高度显著，是价值因子有效性的直接证据。

**A 股特殊注意**：壳价值、涨跌停、停牌、行业集中均会干扰因子构建，
推荐因子数据库：CSMAR、Wind、Tushare Pro。

## 习题参考解答

以下代码对应正文第 7.11 节习题，可直接运行。

In [ ]:
# === 习题 7.1：CAPM 代数计算 ===
print('习题 7.1：CAPM 预期收益与 alpha 计算')
rf_q1 = 0.02; rm_q1 = 0.08; beta_q1 = 1.4; actual_q1 = 0.12
er_q1    = rf_q1 + beta_q1 * (rm_q1 - rf_q1)
alpha_q1 = actual_q1 - er_q1
print(f'CAPM 预期收益 = {rf_q1:.1%} + {beta_q1} x {rm_q1-rf_q1:.1%} = {er_q1:.2%}')
print(f'实际收益 = {actual_q1:.1%}')
print(f'Alpha = {actual_q1:.1%} - {er_q1:.2%} = {alpha_q1:.2%}')
print('alpha > 0：在风险调整后仍有超额表现')

In [ ]:
# === 习题 7.2：四只股票 Beta 与预期收益（真实市场指数）===
print('习题 7.2：四只股票 Beta 与预期收益估计')
mkt_prem_ann = mkt_excess.mean() * TRADING_DAYS

ex72 = []
for ticker in TICKERS:
    res = capm_results[ticker]
    bv  = res.params['MKT']
    ex72.append({
        '股票': ticker,
        'Beta': round(bv, 4),
        'Alpha年化': round(res.params['alpha_const'] * TRADING_DAYS, 4),
        'CAPM预期年化': round(RF_ANNUAL + bv * mkt_prem_ann, 4),
        '实际超额年化': round(excess_rets[ticker].mean() * TRADING_DAYS, 4),
        'R2': round(res.rsquared, 4),
    })
df72 = pd.DataFrame(ex72).set_index('股票')
print(df72.round(4).to_string())
print(f'市场年化超额收益（真实指数）：{mkt_prem_ann:.2%}')
print(f'Beta 最高：{df72["Beta"].idxmax()}')
print(f'CAPM 预期收益最高：{df72["CAPM预期年化"].idxmax()}')
print()
print('参考值：UTILITY≈0.24, BANK≈0.39, LIQUOR≈1.09, TECH≈1.93')

In [ ]:
# === 习题 7.3：R2 与系统性/特质风险 ===
print('习题 7.3：R2 与系统性/特质风险分析')
from fds import annualized_volatility
vol_vals = annualized_volatility(rets)

ex73 = []
for ticker in TICKERS:
    r2 = capm_results[ticker].rsquared
    ex73.append({
        '股票': ticker,
        '年化总波动': round(vol_vals[ticker], 4),
        'R2系统性占比': round(r2, 4),
        '1-R2特质占比': round(1 - r2, 4),
        '系统波动': round(vol_vals[ticker] * np.sqrt(r2), 4),
        '特质波动': round(vol_vals[ticker] * np.sqrt(1 - r2), 4),
    })
df73 = pd.DataFrame(ex73).set_index('股票')
print(df73.to_string())
mv = vol_vals.idxmax()
mr = df73['R2系统性占比'].idxmax()
print(f'\n波动率最高：{mv}，R2 最高：{mr}')
if mv != mr:
    print('结论：高波动率并不等于高 R2。高波动中若特质风险为主，则 R2 反而偏低。')

In [ ]:
# === 习题 7.4：多重共线性影响演示 ===
print('习题 7.4：多重共线性影响（构造高相关因子对比）')

rng2    = np.random.default_rng(99)
T2      = len(excess_f)
f_base  = rng2.normal(0, 0.01, T2)
f_high1 = f_base + rng2.normal(0, 0.003, T2)
f_high2 = f_base + rng2.normal(0, 0.003, T2)
f_indep = rng2.normal(0, 0.01, T2)
y_q4    = excess_f['TECH'].values

X_low  = sm.add_constant(np.column_stack([f_high1, f_indep]))
X_high = sm.add_constant(np.column_stack([f_high1, f_high2]))
res_low  = sm.OLS(y_q4, X_low).fit()
res_high = sm.OLS(y_q4, X_high).fit()

print(f'高相关对（f1 vs f2）：Corr = {np.corrcoef(f_high1, f_high2)[0,1]:.4f}')
print(f'低相关对（f1 vs indep）：Corr = {np.corrcoef(f_high1, f_indep)[0,1]:.4f}')
print()
print('低共线性情况：')
print(f'  f1 t-stat = {res_low.tvalues[1]:.3f},  独立因子 t-stat = {res_low.tvalues[2]:.3f}')
print('高共线性情况：')
print(f'  f1 t-stat = {res_high.tvalues[1]:.3f},  f2 t-stat = {res_high.tvalues[2]:.3f}')
print()
print('结论：高共线性时 t-stat 均下降，各因子独立贡献难以区分。')
print('解决方案：(1) 正交化因子；(2) 删除冗余因子；(3) 岭回归')